In [1]:
import pandas as pd
import numpy as np
import pickle

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

/Users/krish/Downloads/MindScope/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
positive_df = pd.read_csv("../data/positive_emotions.csv")

positive_df.head()

,text,label
0,I finally achieved something I worked so hard for,pride
1,I feel proud of myself today,pride
2,Winning that competition made me feel amazing,pride
3,I never thought I could do it but I did,pride
4,I accomplished my goal and feel fulfilled,pride


In [3]:
model = SentenceTransformer("../models/sentence_model")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5941.03it/s]


In [4]:
positive_embeddings = model.encode(
    positive_df["text"].tolist(),
    show_progress_bar=True
)

positive_embeddings.shape


Batches: 100%|██████████| 1/1 [00:02<00:00,  2.82s/it]


(30, 768)

In [5]:
emotion_centroids = {}

for label in positive_df["label"].unique():

    subset = positive_df[
        positive_df["label"] == label
    ]

    idxs = subset.index.tolist()

    centroid = positive_embeddings[idxs].mean(axis=0)

    emotion_centroids[label] = centroid

In [6]:
with open("../models/positive_centroids.pkl", "wb") as f:
    pickle.dump(emotion_centroids, f)

print("Saved positive emotion centroids")

Saved positive emotion centroids


In [7]:
with open("../models/positive_centroids.pkl", "rb") as f:
    centroids = pickle.load(f)

labels = list(centroids.keys())

matrix = np.array([
    centroids[l] for l in labels
])

In [8]:
def predict_positive_emotion(text):

    vec = model.encode([text])[0]

    sims = cosine_similarity(
        [vec],
        matrix
    )[0]

    idx = np.argmax(sims)

    return labels[idx], sims[idx]

In [9]:
predict_positive_emotion(
    "I finally achieved something important today"
)

('pride', np.float32(0.8951143))